
# 01 — Nettoyage des données météorologiques Silver

## Objectif
Nettoyer, contrôler et fiabiliser les observations météorologiques issues du parsing JSON afin de les rendre exploitables pour les analyses.

## Source
`dbo.raw_meteo_france_parsing` — Données météorologiques structurées.

## Destination
`dbo.silver_meteo_france` — Table Silver du Lakehouse.

## Traitements prévus
- Lecture des données après parsing JSON.
- Contrôle des valeurs manquantes.
- Conversion des températures de Kelvin en Celsius.
- Vérification des coordonnées géographiques.
- Contrôle des valeurs d'humidité.
- Conservation des observations présentant des mesures manquantes.
- Enregistrement des données nettoyées au format Delta.

## Intégration au pipeline
Ce notebook sera exécuté après `NB_02_Parsing_JSON`, dans le pipeline `PL_Meteo_Transformation_journalière`.


In [1]:

# ==================================================
# ÉTAPE 02 — LECTURE DES DONNÉES APRÈS PARSING
# ==================================================

# A — Lire les données parsées depuis le Lakehouse
df_parsing = spark.table("dbo.raw_meteo_france_parsing")

# B — Vérifier le nombre d'observations
print("Nombre de relevés à nettoyer :", df_parsing.count())

# C — Vérifier la structure des données
df_parsing.printSchema()

# D — Afficher les 5 premières observations
display(
    df_parsing.select(
        "observation_date",
        "latitude",
        "longitude",
        "temperature_k",
        "humidite_pct",
        "vitesse_vent",
        "precipitations"
    ).limit(5)
)


StatementMeta(, 19106359-55fe-4093-bef1-08d24e21808a, 3, Finished, Available, Finished, False)

Nombre de relevés à nettoyer : 127130
root
 |-- source: string (nullable = true)
 |-- observation_date: timestamp (nullable = true)
 |-- ingestion_date: timestamp (nullable = true)
 |-- data: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- geo_id_insee: string (nullable = true)
 |-- temperature_k: double (nullable = true)
 |-- humidite_pct: double (nullable = true)
 |-- vitesse_vent: double (nullable = true)
 |-- direction_vent: double (nullable = true)
 |-- precipitations: double (nullable = true)



SynapseWidget(Synapse.DataFrame, cf48a2d5-3810-4257-a60b-bd454c4e5695)

In [2]:

# ==================================================
# ÉTAPE 03 — CONTRÔLE QUALITÉ DES DONNÉES
# ==================================================

from pyspark.sql import functions as F

# A — Définir les colonnes à contrôler
colonnes = [
    "latitude",
    "longitude",
    "temperature_k",
    "humidite_pct",
    "vitesse_vent"
]

# B — Compter les valeurs manquantes
controle_qualite = df_parsing.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in colonnes
])

# C — Afficher le nombre total d'observations
print("Nombre total de relevés :", df_parsing.count())

# D — Afficher les valeurs manquantes
display(controle_qualite)


StatementMeta(, 19106359-55fe-4093-bef1-08d24e21808a, 4, Finished, Available, Finished, False)

Nombre total de relevés : 127130


SynapseWidget(Synapse.DataFrame, 0dd4919e-9315-4efd-8288-8e753fde2139)

In [3]:

# ==================================================
# ÉTAPE 04 — NETTOYAGE DES DONNÉES SILVER
# ==================================================

from pyspark.sql import functions as F

# A — Convertir les températures de Kelvin en Celsius
df_silver = df_parsing.withColumn(
    "temperature_c",
    F.round(F.col("temperature_k") - 273.15, 2)
)

# B — Vérifier les coordonnées géographiques
df_silver = df_silver.filter(
    F.col("latitude").between(-90, 90) &
    F.col("longitude").between(-180, 180)
)

# C — Contrôler les valeurs d'humidité
# Les valeurs manquantes restent NULL
df_silver = df_silver.withColumn(
    "humidite_pct",
    F.when(
        F.col("humidite_pct").between(0, 100),
        F.col("humidite_pct")
    ).otherwise(F.lit(None).cast("double"))
)

# D — Vérifier le nombre d'observations conservées
print("Relevés avant nettoyage :", df_parsing.count())
print("Relevés après nettoyage :", df_silver.count())

# E — Afficher un aperçu des données nettoyées
display(
    df_silver.select(
        "observation_date",
        "latitude",
        "longitude",
        "temperature_c",
        "humidite_pct",
        "vitesse_vent",
        "precipitations"
    ).limit(10)
)


StatementMeta(, 19106359-55fe-4093-bef1-08d24e21808a, 5, Finished, Available, Finished, False)

Relevés avant nettoyage : 127130
Relevés après nettoyage : 127130


SynapseWidget(Synapse.DataFrame, 2a7c271a-a026-414e-b6d0-10886d8b1359)

In [4]:

# ==================================================
# ÉTAPE 05 — ENREGISTREMENT DES DONNÉES SILVER
# ==================================================

# A — Enregistrer les données nettoyées au format Delta
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dbo.silver_meteo_france")

# B — Relire la table Silver enregistrée
df_verification = spark.table("dbo.silver_meteo_france")

# C — Vérifier le nombre d'observations
print("Nombre de relevés Silver :", df_verification.count())

# D — Afficher un aperçu des données
display(
    df_verification.select(
        "observation_date",
        "latitude",
        "longitude",
        "temperature_c",
        "humidite_pct",
        "vitesse_vent",
        "precipitations"
    ).limit(10)
)


StatementMeta(, 19106359-55fe-4093-bef1-08d24e21808a, 6, Finished, Available, Finished, False)

Nombre de relevés Silver : 127130


SynapseWidget(Synapse.DataFrame, 422ce9c7-31ab-4bed-b38f-e553bb2c3789)